# silver_transform  (pipeline task 2)
Builds the Silver Iceberg table `nessie.silver.users_hist` from the Bronze users files
(`users_2020` prepared manually + `users_2021` ingested by dlt in task 1).
Curated (heavy `AboutMe` dropped, typed columns), de-duplicated by user Id and loaded with
`MERGE INTO`. `fecha_cargue` is the load timestamp. Re-running never duplicates rows.

In [1]:
import sys

sys.path.insert(0, "/opt/airflow/jobs")

from pyspark.sql import Window
from pyspark.sql import functions as F

from common import BRONZE_USERS, USERS_HIST, USERS_HIST_LOCATION, expire_old_snapshots, get_spark

spark = get_spark("silver_transform")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/01 16:41:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read Bronze (raw) and normalize column names
dlt and the manual script may name columns differently; both are reduced to lowercase
without underscores before selecting.

In [2]:
raw = spark.read.parquet(*[f"s3a://{p}" for p in BRONZE_USERS.values()])
raw = raw.toDF(*[c.replace("_", "").lower() for c in raw.columns])
raw.createOrReplaceTempView("bronze_users")

26/10/01 16:41:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


## Curate and de-duplicate

In [3]:
curated = spark.sql("""
    SELECT
        CAST(id AS BIGINT)                                  AS id,
        CAST(CAST(reputation AS STRING) AS BIGINT)          AS reputation,
        CAST(creationdate AS TIMESTAMP)                     AS creation_date,
        CAST(displayname AS STRING)                         AS display_name,
        CAST(lastaccessdate AS TIMESTAMP)                   AS last_access_date,
        CAST(views AS INT)                                  AS views,
        CAST(upvotes AS INT)                                AS up_votes,
        CAST(downvotes AS INT)                              AS down_votes,
        NULLIF(CAST(websiteurl AS STRING), '')              AS website_url,
        NULLIF(CAST(location AS STRING), '')                AS location,
        CAST(accountid AS BIGINT)                           AS account_id,
        YEAR(CAST(creationdate AS TIMESTAMP))               AS creation_year
    FROM bronze_users
    WHERE id IS NOT NULL
""")

latest_first = Window.partitionBy("id").orderBy(F.col("last_access_date").desc_nulls_last())
source = (
    curated.withColumn("rn", F.row_number().over(latest_first))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("fecha_cargue", F.current_timestamp())
)
source.createOrReplaceTempView("users_source")

## Create (if missing) and MERGE into Silver

In [4]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {USERS_HIST} (
        id                BIGINT,
        reputation        BIGINT,
        creation_date     TIMESTAMP,
        display_name      STRING,
        last_access_date  TIMESTAMP,
        views             INT,
        up_votes          INT,
        down_votes        INT,
        website_url       STRING,
        location          STRING,
        account_id        BIGINT,
        creation_year     INT,
        fecha_cargue      TIMESTAMP
    ) USING iceberg
    PARTITIONED BY (creation_year)
    LOCATION '{USERS_HIST_LOCATION}'
    TBLPROPERTIES ('format-version' = '2', 'write.parquet.compression-codec' = 'snappy')
""")

spark.sql(f"""
    MERGE INTO {USERS_HIST} t
    USING users_source s
    ON t.id = s.id AND t.creation_year = s.creation_year
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
expire_old_snapshots(spark, USERS_HIST)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|2                       |0                                  |0                                  |2                           |1                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



In [5]:
spark.sql(f"SELECT creation_year, COUNT(*) AS users, MAX(fecha_cargue) AS fecha_cargue FROM {USERS_HIST} GROUP BY creation_year ORDER BY 1").show()
spark.stop()

+-------------+-------+--------------------+
|creation_year|  users|        fecha_cargue|
+-------------+-------+--------------------+
|         2020|1152176|2026-10-01 16:41:...|
|         2021|1364079|2026-10-01 16:41:...|
+-------------+-------+--------------------+

